In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# 1. Initialization and Setup
# Setting a seed ensures we get the same random numbers every time we run the code
np.random.seed(42)

machine_ids=['CNC_01','CNC_02','CNC_03']
start_time=datetime(2026,7,1,8,0) # Simulation starts at July 1st, 08:00 AM
num_records=3000 # Total logs to generate

# 2. Generating Timestamps
# Simualting a sensor reading every 15 minutes 
timestamps= [start_time+ timedelta(minutes=15*i) for i in range (num_records)]

# 3. Generating Sensor Data (Normal Operating Conditions)
# We use normal distribution (loc=mean, scale= standard deviation) to simulate health machines
data={
    'timestamp': timestamps,
    'machine_id': np.random.choice(machine_ids, num_records),
    'spindle_speed_rpm': np.random.normal(loc=12000, scale=150, size=num_records), # Ideal: 12000 RPM
    'vibration_mm_s': np.random.normal(loc=1.2, scale=0.1, size=num_records),      #Ideal: 1.2 mm/s
    'temperature_c': np.random.normal( loc=65.0, scale=3.0, size=num_records)      #Ideal: 65°C
}

# 4. Creating the DataFrame (Our First ERP Table)
df_machine_logs= pd.DataFrame(data)

# Sorting by machine and time to make the time-series-realistic
df_machine_logs= df_machine_logs.sort_values(by=['machine_id', 'timestamp']).reset_index(drop=True)

# Display the result
print("--- Machine Logs Table Created Successfully--")
display(df_machine_logs.head())


--- Machine Logs Table Created Successfully--


,timestamp,machine_id,spindle_speed_rpm,vibration_mm_s,temperature_c
0,2026-07-01 08:15:00,CNC_01,11866.131624,1.150123,60.423146
1,2026-07-01 09:00:00,CNC_01,11873.957785,1.174590,63.921504
2,2026-07-01 09:15:00,CNC_01,11861.575472,1.133709,68.579311
3,2026-07-01 11:00:00,CNC_01,11735.575127,1.272553,61.572214
4,2026-07-01 11:45:00,CNC_01,12013.099659,1.198834,66.465839


In [3]:
# 6. Injecting Anamolies (Simulating Tool Wear iin Tİ-6Al-4V Machining)
# Ti-6Al-4V tends to generate excessive heat. We will introduce a 'Failure' column
df_machine_logs['failure'] = 0

#Rule: If temperature > 85°C  AND vibration > 1.5 mm/s -> Tool failure / Scrap part
anomaly_condition=(df_machine_logs['temperature_c'] > 85.0) & (df_machine_logs['vibration_mm_s'] > 1.5)

# We deliberately manipualte some random rows to meet this failure condition
np.random.seed(42)
anomaly_indices = np.random.choice(df_machine_logs.index, size=int(num_records*0.05), replace=False)  # %5 failure rate

df_machine_logs.loc[anomaly_indices, 'temperature_c'] = np.random.normal(loc=88.00, scale=2.0, size=len(anomaly_indices))
df_machine_logs.loc[anomaly_indices, 'vibration_mm_s'] = np.random.normal(loc=1.7, scale=0.1, size=len(anomaly_indices))
df_machine_logs.loc[anomaly_indices, 'failure'] = 1

# 7. Save the dataset to the 'data' folder
df_machine_logs.to_csv('../data/machine_logs.csv', index=False)

print(f"Data saved, Total records: {len(df_machine_logs)}, Total failures: {df_machine_logs['failure'].sum()}")

Data saved, Total records: 3000, Total failures: 150


In [4]:
# Injecting Missing Values (NaN) randomly into 5% of temperature and vibration readings
np.random.seed(42)
nan_indices_temp= np.random.choice(df_machine_logs.index, size=int(num_records*0.05), replace=False)
nan_indices_vib = np.random.choice(df_machine_logs.index, size=int(num_records * 0.05), replace=False)
nan_indices_machine_id= np.random.choice(df_machine_logs.index, size=50, replace=False)

df_machine_logs.loc[nan_indices_temp, 'temperature_c']=np.nan
df_machine_logs.loc[nan_indices_vib, 'vibration_mm_s']=np.nan
df_machine_logs.loc[nan_indices_machine_id, 'machine_id']= np.nan


# Messing up the 'machine_id' text formatting
df_machine_logs.loc[10:50, 'machine_id']='cnc_01'      #Lowercase typo
df_machine_logs.loc[100:150, 'machine_id']= ' CNC_02 ' #Accidental spaces

# Overwriting the CSV file with the messy data
df_machine_logs.to_csv('../data/machine_logs.csv', index=False)
print("---Data successfully corrupted and saved---")

---Data successfully corrupted and saved---
